In [1]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/processed_milestone2_dataset.xlsx"

df = pd.read_excel(DATA_PATH)
print("Dataset Loaded Successfully!")

TARGET = "on_time_delivery"

# ✅ Fix target FIRST (binary classification)
df[TARGET] = df[TARGET].apply(lambda x: 1 if x >= 1 else 0)
df = df[df[TARGET].isin([0, 1])]

print("Clean Target Distribution:")
print(df[TARGET].value_counts())


Dataset Loaded Successfully!
Clean Target Distribution:
on_time_delivery
0    7137
1    2882
Name: count, dtype: int64


In [3]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10019 entries, 0 to 10018
Data columns (total 36 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   order_id                         10019 non-null  float64       
 1   supplier_id                      10019 non-null  float64       
 2   supplier_rating                  10019 non-null  float64       
 3   supplier_lead_time               10019 non-null  int64         
 4   order_date                       10019 non-null  datetime64[ns]
 5   promised_delivery_date           10019 non-null  datetime64[ns]
 6   actual_delivery_date             10019 non-null  datetime64[ns]
 7   shipping_distance_km             10019 non-null  int64         
 8   order_quantity                   10019 non-null  int64         
 9   unit_price                       10019 non-null  float64       
 10  total_order_value                10019 non-null  float64  

,order_id,supplier_id,supplier_rating,supplier_lead_time,order_date,promised_delivery_date,actual_delivery_date,shipping_distance_km,order_quantity,unit_price,...,region_South,region_West,holiday_period_Yes,carrier_name_DHL,carrier_name_Delhivery,carrier_name_EcomExpress,carrier_name_FedEx,delayed_reason_code_Operational,delayed_reason_code_Traffic,delayed_reason_code_Weather
0,1.0,5322.0,3.4,10,2024-05-15,2024-05-25,2024-05-29,51,48,2153.91,...,False,False,False,False,False,True,False,True,False,False
1,2.0,3932.0,4.3,10,2024-11-12,2024-11-22,2024-11-27,373,91,405.36,...,False,False,True,True,False,False,False,False,False,False
2,3.0,8966.0,3.2,5,2024-08-28,2024-09-02,2024-09-02,1304,25,3241.41,...,True,False,False,False,False,False,False,False,False,False
3,4.0,9832.0,3.9,7,2024-08-12,2024-08-19,2024-08-19,839,71,365.79,...,False,False,False,False,False,False,True,True,False,False
4,5.0,2126.0,3.2,8,2024-07-07,2024-07-15,2024-07-18,258,9,3052.84,...,False,False,False,False,False,False,False,False,False,False


In [28]:
# ❌ ML models cannot handle datetime directly
date_cols = df.select_dtypes(include=["datetime64[ns]"]).columns
print("Dropping datetime columns:", list(date_cols))

df = df.drop(columns=date_cols)


Dropping datetime columns: ['order_date', 'promised_delivery_date', 'actual_delivery_date']


In [29]:
LEAKAGE_COLS = ["delivery_speed", "delayed_reason_code"]

df.drop(
    columns=[c for c in LEAKAGE_COLS if c in df.columns],
    inplace=True
)

print("Leakage columns removed (if present)")


Leakage columns removed (if present)


In [30]:
y = df[TARGET]
X = df.drop(columns=[TARGET])

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)


Feature Shape: (10019, 32)
Target Shape: (10019,)


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Target Distribution:")
print(y_train.value_counts())


Train Target Distribution:
on_time_delivery
0    5709
1    2306
Name: count, dtype: int64


In [32]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print("Class Weights:", class_weights)


Class Weights: {np.int64(0): np.float64(0.7019618146785777), np.int64(1): np.float64(1.7378577623590634)}


In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

lr = LogisticRegression(
    max_iter=2000,
    class_weight=class_weights,
    n_jobs=-1
)

lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)
lr_probs = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, lr_probs))
print("Confusion Matrix (Logistic Regression):")
print(confusion_matrix(y_test, lr_preds))


Logistic Regression ROC-AUC: 0.8050765445066915
Confusion Matrix (Logistic Regression):
[[1003  425]
 [ 140  436]]


In [42]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))
print("Confusion Matrix (Random Forest):")
print(confusion_matrix(y_test, rf_preds))


Random Forest ROC-AUC: 0.9738452672735762
Confusion Matrix (Random Forest):
[[1351   77]
 [  34  542]]


In [38]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

scale_pos_weight = class_weights[0] / class_weights[1]

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best XGBoost Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 400, 'subsample': 0.8}
Best CV ROC-AUC: 0.9774531068595408


In [43]:
best_xgb = grid.best_estimator_

xgb_preds = best_xgb.predict(X_test)
xgb_probs = best_xgb.predict_proba(X_test)[:, 1]

print("XGBoost Test ROC-AUC:", roc_auc_score(y_test, xgb_probs))
print("Confusion Matrix (XGBoost):")
print(confusion_matrix(y_test, xgb_preds))


XGBoost Test ROC-AUC: 0.98166627762216
Confusion Matrix (XGBoost):
[[1421    7]
 [  31  545]]


In [44]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, lr_preds),
        "Precision": precision_score(y_test, lr_preds),
        "Recall": recall_score(y_test, lr_preds),
        "F1": f1_score(y_test, lr_preds),
        "ROC-AUC": roc_auc_score(y_test, lr_probs)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_preds),
        "Precision": precision_score(y_test, rf_preds),
        "Recall": recall_score(y_test, rf_preds),
        "F1": f1_score(y_test, rf_preds),
        "ROC-AUC": roc_auc_score(y_test, rf_probs)
    },
    {
        "Model": "XGBoost (Tuned)",
        "Accuracy": accuracy_score(y_test, xgb_preds),
        "Precision": precision_score(y_test, xgb_preds),
        "Recall": recall_score(y_test, xgb_preds),
        "F1": f1_score(y_test, xgb_preds),
        "ROC-AUC": roc_auc_score(y_test, xgb_probs)
    }
])

results


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.718064,0.506388,0.756944,0.606820,0.805077
1,Random Forest,0.944611,0.875606,0.940972,0.907113,0.973845
2,XGBoost (Tuned),0.981038,0.987319,0.946181,0.966312,0.981666


In [46]:
import joblib

joblib.dump(best_xgb, "best_model.pkl")
joblib.dump(list(X.columns), "model_features.pkl")

print("✅ best_model.pkl saved")
print("✅ model_features.pkl saved")


✅ best_model.pkl saved
✅ model_features.pkl saved


In [49]:
X_test.sample(10).assign(
    probability=best_xgb.predict_proba(X_test.sample(10))[:,1]
)


,order_id,supplier_id,supplier_rating,supplier_lead_time,shipping_distance_km,order_quantity,unit_price,total_order_value,previous_on_time_rate,delivery_days,...,region_West,holiday_period_Yes,carrier_name_DHL,carrier_name_Delhivery,carrier_name_EcomExpress,carrier_name_FedEx,delayed_reason_code_Operational,delayed_reason_code_Traffic,delayed_reason_code_Weather,probability
8313,8314.0,8087.0,2.9,5,986,49,4893.95,239803.55,82.5,10,...,False,False,False,True,False,False,True,False,False,0.003238
7209,7210.0,5894.0,3.3,3,363,74,1939.85,143548.90,85.3,4,...,False,False,True,False,False,False,False,False,False,0.001147
8091,8092.0,5418.0,2.9,3,472,1,327.82,327.82,93.4,5,...,False,False,True,False,False,False,False,False,True,0.003870
3901,3902.0,8781.0,4.1,2,1309,20,2926.32,58526.40,75.8,2,...,False,True,False,False,False,True,False,False,True,0.892022
7152,7153.0,5828.0,3.0,4,1427,29,2552.11,74011.19,78.0,5,...,True,True,False,False,False,False,False,True,False,0.191869
4401,4402.0,5996.0,3.4,8,1381,18,1626.19,29271.42,74.3,9,...,False,True,False,False,False,True,False,False,False,0.004830
8877,8878.0,9671.0,4.4,3,871,86,4701.63,404340.18,96.8,6,...,False,False,False,False,False,True,False,False,False,0.895119
6643,6644.0,6644.0,4.5,3,1067,96,2437.43,233993.28,83.9,3,...,False,False,False,True,False,False,False,True,False,0.024671
8291,8292.0,2848.0,3.4,8,285,80,431.09,34487.20,91.7,7,...,True,False,False,True,False,False,True,False,False,0.964349
5624,5625.0,8718.0,3.6,9,1007,48,4106.36,197105.28,75.9,10,...,False,False,False,True,False,False,True,False,False,0.973109
